# Preparação da área de estudo

Seleção do limite da área de estudo, criação da grelha de referência a 25 m,
rasterização da máscara espacial e alinhamento do Modelo Digital de Elevação.

In [ ]:
import sys
from pathlib import Path

sys.path.append("/code/scripts")

from aoi_utils import align_raster, check_alignment,create_reference, select_aoi

## 1. Configuração

In [ ]:
area = "extremadura" #Opções: "centro" ou "extremadura"
cell_size = 25

if area == "centro":
    aoi_source = "/code/data/raw/limites/caop/Continente_CAOP2025.gpkg"
    aoi_layer = "cont_nuts2"
    aoi_field = "nuts2"
    aoi_value = "Centro"
    target_epsg = 3763
    dem_source = "/code/pen/paper/data/MDT-10m-PT.tif"

elif area == "extremadura":
    aoi_source = "/code/pen/paper/data/lineas_limite/SHP_ETRS89/recintos_autonomicas_inspire_peninbal_etrs89/recintos_autonomicas_inspire_peninbal_etrs89.shp"
    aoi_layer = None
    aoi_field = "NAMEUNIT"
    aoi_value = "Extremadura"
    target_epsg = 25830
    dem_source = "/code/pen/paper/data/lidar_badajoz/dem_lidar_badajoz_clip.tif"

else:
    raise ValueError("Área de Estudo Inválida")
    

## 2. Limte da área de estudo

In [ ]:
base_dir = Path(f"/code/data/processed/{area}")

aoi_out = base_dir / "aoi" / f"{area}.shp"
target_grid = base_dir / "reference" / "target_grid_25m.tif"
aoi_mask = base_dir / "reference" / "aoi_mask_25m.tif"
dem_out = base_dir / "topo" / "derived" / f"dem_{area}_clip.tif"

for path in [aoi_out, target_grid, aoi_mask, dem_out]:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.unlink(missing_ok=True)

In [ ]:
aoi = select_aoi(
    source=aoi_source,
    layer=aoi_layer,
    field=aoi_field,
    value=aoi_value,
    epsg=target_epsg,
    output=str(aoi_out)
)

aoi.plot()

## 3. Grelha de referência, máscara e DEM

In [ ]:
create_reference(
    aoi=str(aoi_out),
    grid=str(target_grid),
    mask=str(aoi_mask),
    cellsize=cell_size,
    epsg=target_epsg
)

In [ ]:
align_raster(
    raster=dem_source,
    reference=str(target_grid),
    clip=str(aoi_out),
    output=str(dem_out)
)

In [ ]:
check_alignment(
    rasters=[str(aoi_mask), str(dem_out)],
    reference=str(target_grid)
)

print("Limite:", aoi_out)
print("Grelha:", target_grid)
print("Máscara:", aoi_mask)
print("DEM:", dem_out)